In [1]:

# Run once. After it finishes, RESTART the kernel, then run all cells again.
import subprocess, sys

pkgs = [
    "pip install --quiet --upgrade pip",
    "pip uninstall -y transformers tokenizers torchvision bitsandbytes peft accelerate trl sentence-transformers 2>/dev/null || true",
    "pip install --quiet transformers==4.37.2 tokenizers==0.15.2",
    "pip install --quiet bitsandbytes==0.42.0",
    "pip install --quiet peft==0.7.1 accelerate==0.26.1",
    "pip install --quiet sentence-transformers==2.7.0",
    "pip install --quiet datasets huggingface_hub",
    "pip install --quiet scipy scikit-learn pandas matplotlib pillow",
]
for cmd in pkgs:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = "✓" if r.returncode == 0 else "✗"
    print(f"{status} {cmd[:60]}")

print("\nDone. RESTART KERNEL NOW then run all cells.")


✓ pip install --quiet --upgrade pip
✓ pip uninstall -y transformers tokenizers torchvision bitsand
✓ pip install --quiet transformers==4.37.2 tokenizers==0.15.2
✓ pip install --quiet bitsandbytes==0.42.0
✓ pip install --quiet peft==0.7.1 accelerate==0.26.1
✓ pip install --quiet sentence-transformers==2.7.0
✓ pip install --quiet datasets huggingface_hub
✓ pip install --quiet scipy scikit-learn pandas matplotlib pil

Done. RESTART KERNEL NOW then run all cells.


In [2]:
import os, gc, json, warnings, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import stats as scipy_stats
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")
print(f"torch      {torch.__version__}")
print(f"CUDA       {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
import transformers, peft, bitsandbytes
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print("✓ All imports OK")

2026-06-09 06:18:50.126634: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780985930.296524      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780985930.345447      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780985930.743399      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780985930.743439      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780985930.743442      58 computation_placer.cc:177] computation placer alr

torch      2.10.0+cu128
CUDA       True | device: Tesla P100-PCIE-16GB
False

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {PosixPath('/usr/local/lib/python3.12/dist-packages/cv2/../../lib64')}
The following directories listed in your path were found to be non-existent: {PosixPath('//www.kaggle.com'), PosixPath('https')}
The following directories listed in your path were found to be non-existent: {PosixPath('//colab\\.(sandbox|research)\\.google\\.com'), PosixPath('https')}
The following directories listed in your path were found to be non-existent: {PosixPath('//dp.kaggle.net'), PosixPath('https')}
The following directories listed in your path were found to be non-existent: {PosixPath('/kaggle/lib/kagglegym')}
The following directories listed in your path were found to be non-existent: {PosixPath('gcr.io/kaggle-gpu-images/python@sha256'), PosixPath('57e612b484cf3df5026ee4dc

RuntimeError: 
        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

In [ ]:
import os

WORK_DIR  = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
CACHE_DIR = os.path.join(WORK_DIR, "llava_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

CFG = {
    # Model
    "MODEL_ID":            "llava-hf/llava-1.5-7b-hf",
    "LORA_RANK":           16,
    # Training
    "PPO_STEPS":           400,
    "TRAIN_ITEMS":         400,
    "LEARNING_RATE":       1e-5,
    "TEMPERATURE":         0.9,
    "MAX_NEW_TOKENS":      50,
    "EMA_DECAY":           0.9,
    "GRAD_CLIP":           0.5,
    # PPO-Clip
    "PPO_EPSILON":         0.2,
    # Evaluation
    "TEST_ITEMS":          500,
    "EVAL_TEMPERATURE":    0.7,
    # Seeds (8 seeds)
    "SEEDS":               [42, 123, 7, 1, 2, 3, 4, 5],
    # Reward
    "W_TASK":              1.0,
    "W_FACT":              0.5,
    "PENALTY":            -0.5,
    "FGM_THRESHOLD":       0.6,
    "TASK_THRESHOLD":      0.7,
    # Stats
    "CI_LEVEL":            0.95,
}

print("CFG ready:")
for k,v in CFG.items():
    print(f"  {k:<25} {v}")

In [ ]:
from huggingface_hub import snapshot_download

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def nuke(*extra_names):
    """Free all known model variables from GPU memory."""
    names = list(extra_names) + [
        "model","base","sm_reinforce","sm_ppoclip","sm_hra",
        "vm","vanilla_model","lora_model"
    ]
    for name in names:
        if name in globals():
            try:    globals()[name].cpu()
            except: pass
            try:    del globals()[name]
            except: pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")

# Download weights once (file transfer only — no GPU)
already_cached = any(
    f.endswith(".safetensors")
    for _, _, files in os.walk(CACHE_DIR)
    for f in files
)

if already_cached:
    print(f"✓ Model weights cached at {CACHE_DIR}")
else:
    print(f"Downloading model weights to {CACHE_DIR} ...")
    print("  (One-time download ~14GB — subsequent runs load from disk)")
    snapshot_download(
        repo_id=CFG["MODEL_ID"],
        cache_dir=CACHE_DIR,
        ignore_patterns=["*.msgpack","*.h5","flax_model*","tf_model*"],
    )
    already_cached = True
    print("✓ Download complete")

print(f"\nWorking dir : {WORK_DIR}")
print(f"Cache dir   : {CACHE_DIR}")
print(f"Cached      : {already_cached}")

In [ ]:
def _bnb():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
        bnb_4bit_quant_type="nf4",
    )

def _lora():
    return LoraConfig(
        r=CFG["LORA_RANK"],
        lora_alpha=CFG["LORA_RANK"] * 2,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj","v_proj"],
        task_type="CAUSAL_LM",
    )

def make_lora_model():
    """Load 4-bit LLaVA + fresh LoRA. Loads from local cache."""
    gc.collect(); torch.cuda.empty_cache()
    free, _ = torch.cuda.mem_get_info()
    print(f"  VRAM before load: {free/1024**3:.2f} GB")
    base = LlavaForConditionalGeneration.from_pretrained(
        CFG["MODEL_ID"],
        quantization_config=_bnb(),
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    m = get_peft_model(base, _lora())
    m.print_trainable_parameters()
    free2, _ = torch.cuda.mem_get_info()
    print(f"  VRAM after load:  {free2/1024**3:.2f} GB")
    return m

def make_vanilla_model():
    """Load 4-bit LLaVA with no LoRA."""
    gc.collect(); torch.cuda.empty_cache()
    m = LlavaForConditionalGeneration.from_pretrained(
        CFG["MODEL_ID"],
        quantization_config=_bnb(),
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    free, _ = torch.cuda.mem_get_info()
    print(f"  Vanilla loaded. VRAM: {free/1024**3:.2f} GB")
    return m

print("✓ Model factory ready (make_lora_model / make_vanilla_model)")